# 02 — SC-FEGAN Smoke Test

**목표**: SC-FEGAN을 자동 생성한 마스크/스케치/컬러 입력으로 1장 추론해서 결과가 나오는지 확인.

**전제**:
- `01_env_check.ipynb` 완료
- ckpt 파일이 Google Drive `cv-final/SC-FEGAN/ckpt/` 에 업로드됨
  - **필수 2개**: `SC-FEGAN.ckpt.data-00000-of-00001`, `SC-FEGAN.ckpt.index`
  - **선택 1개**: `SC-FEGAN.ckpt.meta` (그래프는 model.py에서 빌드하므로 없어도 동작)
  - 다운로드: https://drive.google.com/open?id=1VPsYuIK_DY3Gw07LEjUhg2LwbEDlFpq1
  - 다운로드 받은 후 본인의 MyDrive로 복사

**Week 1 DoD**: 샘플 1장에 대해 결과 이미지 1장이 그럴듯하게 나오면 통과.

## 1. 셋업 (01에서 이어짐)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Keras 2 강제 — SC-FEGAN의 tf.layers.conv2d 는 Keras 2 전용
# (TF 2.16+ 기본 Keras 3에는 conv2d 가 제거됨)
# ─────────────────────────────────────────────────────────────
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!pip install --quiet tf-keras

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
SCFEGAN_DIR = f'{PROJECT_ROOT}/SC-FEGAN'
CKPT_DIR = f'{SCFEGAN_DIR}/ckpt/SC-FEGAN.ckpt'

import sys
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, SCFEGAN_DIR)

# ckpt 존재 확인 (.data + .index만 필수, .meta는 선택)
REQUIRED = ['data-00000-of-00001', 'index']
OPTIONAL = ['meta']

for ext in REQUIRED:
    p = f'{CKPT_DIR}.{ext}'
    assert os.path.exists(p), f'❌ 필수 파일 누락: {p}'
    print(f'✅ {os.path.basename(p)} ({os.path.getsize(p)/1024/1024:.1f} MB)')

for ext in OPTIONAL:
    p = f'{CKPT_DIR}.{ext}'
    if os.path.exists(p):
        print(f'✅ {os.path.basename(p)} (선택 파일, 발견됨)')
    else:
        print(f'⚠️  {os.path.basename(p)} 없음 — model.py가 그래프를 직접 빌드하므로 무시 가능')

## 2. SC-FEGAN 모델 로드

원본 demo.yaml은 `./ckpt/SC-FEGAN.ckpt`을 가리키므로 임시로 yaml 생성.

In [ ]:
%cd $SCFEGAN_DIR

# Colab에서 demo.yaml 경로를 절대경로로 덮어쓰기
with open('demo.yaml', 'w') as f:
    f.write(f'''INPUT_SIZE: 512
BATCH_SIZE: 1
GPU_NUM: 0
CKPT_DIR: '{CKPT_DIR}'
''')

# ─────────────────────────────────────────────────────────────
# yaml.load(f) 호환 패치 (PyYAML 6+ 는 Loader 필수)
# ─────────────────────────────────────────────────────────────
import yaml as _yaml
_yaml_load_orig = _yaml.load
def _yaml_load_compat(stream, Loader=None):
    if Loader is None:
        Loader = _yaml.SafeLoader
    return _yaml_load_orig(stream, Loader=Loader)
_yaml.load = _yaml_load_compat
print('✅ yaml.load compat 패치 OK')

# ─────────────────────────────────────────────────────────────
# TF 2.x compat 셋업 + tf.contrib 모듈 stub
# (Python 3.12 환경에서 TF 1.x 설치 불가 → TF 2.x로 동작시킴)
# SC-FEGAN의 tf.contrib.framework.load_variable, tf.contrib.slim.nets,
# add_arg_scope를 모두 모방한다.
# ─────────────────────────────────────────────────────────────
import sys, types
import tensorflow as tf

if tf.__version__.startswith('2.'):
    import tensorflow.compat.v1 as tf_v1
    tf_v1.disable_v2_behavior()
    sys.modules['tensorflow'] = tf_v1
    tf = tf_v1

# tf.contrib 가짜 모듈 트리 생성
_contrib = types.ModuleType('tensorflow.contrib')
_framework = types.ModuleType('tensorflow.contrib.framework')
_fw_py = types.ModuleType('tensorflow.contrib.framework.python')
_fw_ops = types.ModuleType('tensorflow.contrib.framework.python.ops')
_slim = types.ModuleType('tensorflow.contrib.slim')
_slim_nets = types.ModuleType('tensorflow.contrib.slim.nets')

def _load_variable(ckpt_path, name):
    """tf.contrib.framework.load_variable 대체 — 가중치 파일에서 텐서 한 개 로드."""
    reader = tf.train.NewCheckpointReader(ckpt_path)
    return reader.get_tensor(name.rsplit(':', 1)[0])

def _add_arg_scope(fn):
    """tf.contrib.framework.add_arg_scope 대체 — no-op 데코레이터."""
    return fn

_framework.load_variable = _load_variable
_fw_ops.add_arg_scope = _add_arg_scope
_contrib.framework = _framework
_contrib.slim = _slim
_slim.nets = _slim_nets

sys.modules['tensorflow.contrib'] = _contrib
sys.modules['tensorflow.contrib.framework'] = _framework
sys.modules['tensorflow.contrib.framework.python'] = _fw_py
sys.modules['tensorflow.contrib.framework.python.ops'] = _fw_ops
sys.modules['tensorflow.contrib.slim'] = _slim
sys.modules['tensorflow.contrib.slim.nets'] = _slim_nets
tf.contrib = _contrib

print(f'✅ TF setup OK: version={tf.__version__}, tf.contrib stubbed')

In [ ]:
from utils.config import Config
from model import Model

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
config = Config('demo.yaml')
model = Model(config)
model.load_demo_graph(config)
print('✅ SC-FEGAN 모델 로드 완료')

## 3. 자동 입력 생성 (우리 모듈)

In [ ]:
import cv2, numpy as np
from project.face_analyzer import detect_landmarks, normalize_face, compute_ratios
from project.recommender import recommend, load_procedures
from project.input_generator import make_mask

img = cv2.imread('samples/00001.png')
aligned, meta = normalize_face(img)
lm = detect_landmarks(aligned)
ratios = compute_ratios(lm)

procs = load_procedures(f'{PROJECT_ROOT}/project/recommender/procedures.yaml')
recs = recommend(ratios, procs)
print(f'추천 {len(recs)}건')
for r in recs:
    print(f'  [{r.confidence:.2f}] {r.name_ko}')

# 1순위 시술의 마스크 생성
target = recs[0]
mask = make_mask(lm, target.procedure)  # (512,512,1) uint8 0/255
print('mask:', mask.shape, mask.dtype, 'nonzero=', int((mask>0).sum()))

In [ ]:
# Week 1은 자동 스케치 없이 빈 스케치로 smoke test (Week 2에서 sketch.py 추가 예정)
sketch = np.zeros((512, 512, 1), dtype=np.uint8)

# 컬러: 마스크 외부 평균 피부톤
skin = aligned[mask[..., 0] == 0]
avg_bgr = np.median(skin, axis=0).astype(np.uint8)
stroke = np.zeros_like(aligned)
stroke[mask[..., 0] > 0] = avg_bgr
print('avg skin BGR:', avg_bgr)

## 4. SC-FEGAN 추론

9채널 텐서 빌드 → `model.demo(config, batch)` 호출.

In [ ]:
# 정규화
img_n = aligned.astype(np.float32) / 127.5 - 1
stroke_n = stroke.astype(np.float32) / 127.5 - 1
mask_n = (mask > 0).astype(np.float32)  # 0/1
sketch_n = sketch.astype(np.float32)    # 0/1 (지금은 전부 0)

noise = np.random.randint(0, 256, (512, 512, 1), dtype=np.uint8).astype(np.float32) / 255
noise = noise * mask_n  # 마스크 내부만
sketch_n = sketch_n * mask_n
stroke_n = stroke_n * mask_n

batch = np.concatenate([
    img_n[None],      # 0:3  RGB image (BGR이지만 학습 시 동일 포맷)
    sketch_n[None],   # 3:4  sketch
    stroke_n[None],   # 4:7  color
    mask_n[None],     # 7:8  mask
    noise[None]       # 8:9  noise
], axis=3)
print('batch shape:', batch.shape, 'dtype:', batch.dtype)

import time
t0 = time.time()
out = model.demo(config, batch)
print(f'추론 시간: {time.time()-t0:.2f}s')
print('output shape:', out.shape)

result = ((out[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
print('result:', result.shape, result.dtype)

## 5. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(aligned[..., ::-1]); axes[0].set_title('Before (Aligned)')
axes[1].imshow(mask[..., 0], cmap='gray'); axes[1].set_title(f'Mask ({target.name_ko})')
axes[2].imshow(stroke[..., ::-1]); axes[2].set_title('Color stroke')
axes[3].imshow(result[..., ::-1]); axes[3].set_title('After (SC-FEGAN)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/outputs/smoke_test_01.png', dpi=150)
plt.show()

## 6. 결과 평가 체크리스트

- [ ] 추론이 에러 없이 완료
- [ ] 결과 이미지가 정상 범위([0, 255]) uint8
- [ ] 마스크 영역 바깥은 원본과 거의 동일
- [ ] 마스크 영역 내부에 SC-FEGAN이 생성한 픽셀이 그럴듯한지 (눈으로 확인)

**FAIL 시 대응**:
- ckpt 로드 실패 → 파일 경로/권한 확인
- TF version mismatch → 01 노트북의 옵션 B로 재설치
- 결과가 완전 깨짐 → 입력 정규화 범위 재확인 ([-1, 1])
- 결과가 원본과 동일 → mask가 0이거나 model 가중치 미로딩

→ 다음 Week 2: sketch.py + 슬라이더 기능 추가